In [5]:
import sys
import math
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from src.simulation import (
    simulate_sir,
    choose_immunized_random,
    choose_immunized_by_degree,
    choose_immunized_by_eigenvector_centrality,
    choose_immunized_by_betweenness_centrality,
    choose_immunized_by_adaptive_degree,
    choose_immunized_by_highest,
    choose_immunized_by_neighbor_nomination,
)
from src.data_processing import load_wy_county_graphs


In [6]:
# Config + data discovery
BETA = 0.03
GAMMA = 0.03
STEPS = 200
N_INFECTED = 5
K_PERCENTAGES = list(range(0, 81, 10))
N_TRIALS = 10
BASE_SEED = 20260308

strategy_functions = {
    "Random": choose_immunized_random,
    "Degree": choose_immunized_by_degree,
    "Eigenvector": choose_immunized_by_eigenvector_centrality,
    "Betweenness": choose_immunized_by_betweenness_centrality,
    "Adaptive Degree": choose_immunized_by_adaptive_degree,
    "Highest": choose_immunized_by_highest,
    "Neighbor Nomination": choose_immunized_by_neighbor_nomination,
}

WY_BASE = Path("../data/WY")
county_dirs = sorted([p for p in WY_BASE.iterdir() if p.is_dir() and p.name.isdigit()])
print(f"Counties found: {len(county_dirs)}")
print("First 5:", [d.name for d in county_dirs[:5]])


Counties found: 23
First 5: ['56001', '56003', '56005', '56007', '56009']


In [7]:
# Quick data-loading sanity check (single county)
if not county_dirs:
    raise RuntimeError("No county folders found under ../data/WY")

sample_county = county_dirs[0]
B_sample, G_sample = load_wy_county_graphs(sample_county)
print("Sample county:", sample_county.name)
print("Bipartite nodes/edges:", B_sample.number_of_nodes(), B_sample.number_of_edges())
print("Person graph nodes/edges:", G_sample.number_of_nodes(), G_sample.number_of_edges())


Sample county: 56001
Bipartite nodes/edges: 43254 52417
Person graph nodes/edges: 28901 2758799


In [8]:
def stable_seed(*parts):
    key = "|".join(str(part) for part in parts)
    digest = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2**32)


def run_single_trial(G, county_id, strategy_name, strategy_fn, k_count, trial_idx):
    seed0 = stable_seed(BASE_SEED, county_id, strategy_name, k_count, trial_idx)

    random.seed(seed0)
    immunized = strategy_fn(G, k_count)

    eligible = [node for node in G.nodes if node not in immunized]
    n_start = min(N_INFECTED, len(eligible))
    start_rng = random.Random(seed0 + 1)
    initial_infected = start_rng.sample(eligible, n_start) if n_start > 0 else []

    return simulate_sir(
        G,
        beta=BETA,
        gamma=GAMMA,
        steps=STEPS,
        initial_infected=initial_infected,
        immunized=immunized,
        rng=np.random.default_rng(seed0 + 2),
    )


def run_county_experiment(county_dir):
    county_id = county_dir.name
    _, G = load_wy_county_graphs(county_dir)
    pop = G.number_of_nodes()
    rows = []

    for k_pct in K_PERCENTAGES:
        k_count = math.floor(k_pct * pop / 100)

        for strategy_name, strategy_fn in strategy_functions.items():
            peaks, totals = [], []
            for trial_idx in range(N_TRIALS):
                out = run_single_trial(G, county_id, strategy_name, strategy_fn, k_count, trial_idx)
                peaks.append(max(out["I"]))
                totals.append(out["Total_inf"])

            rows.append({
                "county": county_id,
                "nodes": pop,
                "k_pct": k_pct,
                "k_count": k_count,
                "strategy": strategy_name,
                "peak_mean": float(np.mean(peaks)),
                "peak_std": float(np.std(peaks)),
                "total_mean": float(np.mean(totals)),
                "total_std": float(np.std(totals)),
            })

    return pd.DataFrame(rows)


In [ ]:
# Run all counties (robust if run directly)
if "county_dirs" not in globals() or not county_dirs:
    WY_BASE = Path("../data/WY")
    county_dirs = sorted([p for p in WY_BASE.iterdir() if p.is_dir() and p.name.isdigit()])

all_results = []
for idx, county_dir in enumerate(county_dirs, start=1):
    print(f"[{idx}/{len(county_dirs)}] Running county {county_dir.name}...")
    all_results.append(run_county_experiment(county_dir))

results_df = pd.concat(all_results, ignore_index=True)
print("Done:", results_df.shape)
results_df.head()


In [ ]:
# Summary + one-county plot
summary = (
    results_df.groupby(["county", "k_pct", "strategy"], as_index=False)
    .agg(peak_mean=("peak_mean", "mean"), total_mean=("total_mean", "mean"))
)

ranking = (
    summary.groupby("strategy", as_index=False)
    .agg(avg_peak=("peak_mean", "mean"), avg_total=("total_mean", "mean"))
    .sort_values("avg_peak")
)
print(ranking)

county_to_plot = county_dirs[0].name
subset = summary[summary["county"] == county_to_plot]

plt.figure(figsize=(10, 6))
for strategy_name in strategy_functions:
    s = subset[subset["strategy"] == strategy_name]
    plt.plot(s["k_pct"], s["peak_mean"], marker="o", label=strategy_name)

plt.title(f"County {county_to_plot}: Peak infected vs % immunized (MC mean)")
plt.xlabel("Percent immunized")
plt.ylabel("Peak infected")
plt.legend()
plt.tight_layout()
plt.show()
